# Safe proxy and browser setup for `juscraper-cloud`

This notebook demonstrates vendor-neutral configuration. It is safe to execute as published: network calls to judicial systems remain disabled unless `JUSCRAPER_RUN_LIVE=1` is set deliberately. Keep CNJs, credentials, endpoint URLs, headers, and response bodies in private storage. In a fresh notebook runtime, install the package first with `%pip install \"juscraper-cloud[browser]\"`, then restart the kernel before continuing.

In [ ]:
import json
import os

from juscraper_cloud import (
    CdpPlaywrightProvider,
    LocalPlaywrightProvider,
    PlaywrightWsProvider,
    ProxyRoute,
    scraper,
)

RUN_LIVE = os.environ.get("JUSCRAPER_RUN_LIVE") == "1"
print("Live judicial requests enabled:", RUN_LIVE)

## Proxy route and custom CA bundle

Set `JUSCRAPER_PROXY_URL` in a private secret store. If the proxy uses a private certificate authority, set `JUSCRAPER_PROXY_CA_BUNDLE` to a PEM bundle. TLS verification stays enabled.

In [ ]:
proxy_route = ProxyRoute.from_env()
print("Proxy route configured:", proxy_route is not None)

if RUN_LIVE:
    if proxy_route is None:
        raise RuntimeError("Configure the proxy route before enabling a live proxy example")
    private_court = os.environ["JUSCRAPER_TARGET_COURT"]
    private_query = os.environ["JUSCRAPER_PRIVATE_QUERY"]
    proxy_client = scraper(private_court, route=proxy_route, sleep_time=1.0)
    proxy_result = proxy_client.cjsg(private_query, paginas=range(1, 2))
    print("Proxy query completed; rows are retained only in memory:", len(proxy_result))
else:
    print("Dry run: no proxy request was made.")

## Local Playwright, Playwright WebSocket, or CDP

Choose one mode with `JUSCRAPER_BROWSER_MODE`: `local`, `ws`, or `cdp`. Remote endpoints come from `JUSCRAPER_BROWSER_WS_ENDPOINT` or `JUSCRAPER_BROWSER_CDP_ENDPOINT`. Optional secret headers come from `JUSCRAPER_BROWSER_HEADERS_JSON`. The cell reports only whether configuration exists; it never prints its values.

In [ ]:
mode = os.environ.get("JUSCRAPER_BROWSER_MODE", "local")
headers_raw = os.environ.get("JUSCRAPER_BROWSER_HEADERS_JSON")
headers = json.loads(headers_raw) if headers_raw else None

if mode == "local":
    browser = LocalPlaywrightProvider(headless=True)
elif mode == "ws":
    browser = PlaywrightWsProvider(os.environ["JUSCRAPER_BROWSER_WS_ENDPOINT"], headers=headers)
elif mode == "cdp":
    browser = CdpPlaywrightProvider(os.environ["JUSCRAPER_BROWSER_CDP_ENDPOINT"], headers=headers)
else:
    raise ValueError("JUSCRAPER_BROWSER_MODE must be local, ws, or cdp")

print("Browser mode configured:", mode)

A custom provider only needs a managed synchronous `open_page()` context. Keep private provisioning code outside this notebook and pass that object to `scraper` in the same way.

In [ ]:
if RUN_LIVE:
    private_court = os.environ["JUSCRAPER_TRF_COURT"]
    private_deployment = os.environ["JUSCRAPER_TRF_DEPLOYMENT"]
    private_cnj = os.environ["JUSCRAPER_PRIVATE_CNJ"]
    if private_court not in {"trf2", "trf4"}:
        raise ValueError("JUSCRAPER_TRF_COURT must select a documented browser court")
    browser_client = scraper(private_court, browser=browser, deployment=private_deployment, sleep_time=1.0)
    browser_result = browser_client.cpopg(private_cnj)
    print("Browser query completed; rows are retained only in memory:", len(browser_result))
else:
    print("Dry run: no browser request was made.")

## Deployment, privacy, and rate-limit checklist

- TRF2 requires `rj` or `es`; TRF4 requires `pr`, `sc`, or `rs`.
- Keep raw CNJs, proxy URLs, endpoint URLs, authentication headers, and responses private.
- Do not print or save response bodies in a public notebook.
- Keep `sleep_time >= 1.0`, run sequentially, and start with a small sample.
- A remote route must fail closed; do not retry directly after a transport failure.